# ⚖️ Aula 22 — Ética, Segurança Industrial e Responsabilidade**Disciplina:** IA Aplicada à Engenharia Química**Dataset:** torre_resfriamento_12m.csv — 12 meses da torre de resfriamento---## A pergunta central"Se o modelo errar e o erro custar uma vida — quem é responsável?"Modelos PREDIZEM e OTIMIZAM. Mas aqui o "output" não é um número: é uma RESPONSABILIDADE.

## Viés em dados de processo- **Sazonalidade:** 2 meses de verão → modelo falha no inverno- **Regime operacional:** só operação estável → falha em start-ups/transições- **Detecção:** comparar distribuição treino vs produção (PSI / KS-test)PSI = Σ (pct_treino − pct_prod) · ln(pct_treino/pct_prod)PSI < 0.1 estável | 0.1-0.25 alerta | > 0.25 drift significativo

## 3.1 — Exercício Guiado: Auditar viés por sazonalidade

### Passo 1: carregar dados da torre

In [ ]:
import pandas as pd, numpy as npURL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula22/torre_resfriamento_12m.csv"df = pd.read_csv(URL, parse_dates=['timestamp'])df.groupby('estacao')['T_entrada_C'].agg(['mean','min','max']).round(1)

### Passo 2: separar treino (verao, estacao 0) e producao (inverno, estacao 2)

In [ ]:
treino = df[df['estacao']==0]['T_entrada_C'].values   # verao (quente)producao = df[df['estacao']==2]['T_entrada_C'].values  # inverno (frio)print(f"Treino (verao):  n={len(treino)}, media={treino.mean():.1f}C")print(f"Producao (inv):  n={len(producao)}, media={producao.mean():.1f}C")

### Passo 3: calcular PSI (drift)

In [ ]:
def psi(a, b, bins=10):    pa, _ = np.histogram(a, bins=bins, density=True)    pb, _ = np.histogram(b, bins=bins, density=True)    pa = pa/pa.sum()+1e-6; pb = pb/pb.sum()+1e-6    return np.sum((pa-pb)*np.log(pa/pb))p = psi(treino, producao)print(f"PSI(T_entrada) verao vs inverno = {p:.2f}")if p < 0.1: print("-> estavel")elif p < 0.25: print("-> alerta")else: print("-> DRIFT SIGNIFICATIVO: modelo treinado no verao não generaliza p/ inverno")

### Passo 4: visualizar sobreposição de distribuicoes

In [ ]:
import matplotlib.pyplot as pltplt.hist(treino, bins=20, alpha=0.5, label='Treino verao')plt.hist(producao, bins=20, alpha=0.5, label='Producao inverno')plt.xlabel('T_entrada (C)'); plt.ylabel('freq')plt.legend(); plt.grid(alpha=0.3); plt.title('Viés por sazonalidade')plt.tight_layout(); plt.show()

### Passo 5: posicionar a IA na hierarquia de riscos

In [ ]:
# Hierarquia de controle de riscos (IA no nivel 3 = suporte, NUNCA barreira final)print('1. Eliminar o risco')print('2. Solucao de engenharia (alivio de pressao, instrumentacao de seguranca)')print('3. IA/ML como SUPORTE (recomendacoes, alarmes)')print('4. Procedimentos administrativos / permissao de trabalho')print('5. EPI')print()print('Mitigacao: operar o modelo como SUPORTE com validacao humana no inverno,')print('e monitorar drift antes de qualquer decisao automatica.')

> **Conclusao:** modelo treinado só no verão sofre viés por sazonalidade (PSI 0.61).> Não deve decidir sozinho — deve operar como suporte com validação humana.

## 3.2 — Debate: Soft-sensor erra 2%4 papéis: (A) Engenheiro de IA quer implantar | (B) Operador não confia |(C) Gerente quer custo | (D) Auditor quer barreiras.- Erro p/ baixo = multa R$50k | erro p/ cima = alarme falso → perde confiança- Negocie salvaguardas: quem assina? quais barreiras?"

## Checklist de Segurança- [ ] Responsável nomeado (ART / política)- [ ] Procedimento de exceção- [ ] Barreira de proteção instrumentada independente- [ ] Drift monitorado (PSI)- [ ] Log/registro da decisão- [ ] Política interna de uso de IA escrita